In [96]:
import torch
print(torch.__version__)

2.7.1+cu128


In [97]:
print(torch.cuda.is_available())

True


In [98]:
print(torch.tensor([1, 2, 3]))
print(torch.tensor([[1, 2, 3], [4, 5, 6]]))
print(torch.LongTensor([1, 2, 3]))
print(torch.FloatTensor([1, 2, 3]))


tensor([1, 2, 3])
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([1, 2, 3])
tensor([1., 2., 3.])


In [99]:
tensor = torch.rand((3, 3), dtype = torch.float)
print(tensor)

tensor([[0.6195, 0.2391, 0.2689],
        [0.3315, 0.3122, 0.2912],
        [0.3652, 0.6299, 0.0954]])


In [100]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = torch.cuda.FloatTensor([1, 2, 3])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor = torch.rand((1, 1), device = device)
print(cpu)
print(gpu)

tensor([1., 2., 3.])
tensor([1., 2., 3.], device='cuda:0')


In [101]:
cpu = torch.FloatTensor([1, 2, 3])
gpu = cpu.cuda()
gpu2cpu = gpu.cpu()
cpu2gpu = cpu.to('cuda')

In [102]:
import numpy as np
ndarray = np.array([1, 2, 3], dtype = np.uint8)
print(torch.tensor(ndarray))
print(torch.Tensor(ndarray))
print(torch.from_numpy(ndarray))

tensor([1, 2, 3], dtype=torch.uint8)
tensor([1., 2., 3.])
tensor([1, 2, 3], dtype=torch.uint8)


In [103]:
tensor = torch.cuda.FloatTensor([1, 2, 3])
ndarray = tensor.detach().cpu().numpy()
print(ndarray)
print(type(ndarray))


[1. 2. 3.]
<class 'numpy.ndarray'>


In [104]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split

In [105]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x = df.iloc[:, 0].values
        self.y = df.iloc[:, 1].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x[index] ** 2, self.x[index]])
        y = torch.FloatTensor([self.y[index]])
        return x, y

    def __len__(self):
        return self.length

In [106]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(2, 1)

    def forward(self, x):
        x = self.layer(x)
        return x

In [107]:
dataset = CustomDataset('./dataset/non_linear.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
train_dataloader = DataLoader(train_dataset, batch_size = 16, shuffle = True, drop_last = True)
val_dataloader = DataLoader(val_dataset, batch_size = 4, shuffle = True, drop_last = True)
test_dataloader = DataLoader(test_dataset, batch_size = 4, shuffle = False, drop_last = True)

In [108]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.MSELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [109]:
checkpoint = 0
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)
        torch.save(model, f'./models/checkpoint-{checkpoint}.pt')
        checkpoint += 1


100 tensor(0.0857, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(0.0821, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(0.0803, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.0809, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.0809, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.0812, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.0793, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.0790, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.0784, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.0779, device='cuda:0', grad_fn=<DivBackward0>)


In [110]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
    outputs = model(x)
    print(outputs)

tensor([[22.5025],
        [ 9.0274],
        [55.6473],
        [ 9.5594]], device='cuda:0')


In [111]:
torch.save(model, './models/model.pt')

In [112]:
torch.save(model.state_dict(), './models/model_state_dict.pt')

In [113]:
model = torch.load('./models/model.pt', map_location = device, weights_only = False)

In [114]:
print(model)

CustomModel(
  (layer): Linear(in_features=2, out_features=1, bias=True)
)


In [115]:
model_state_dict = torch.load('./models/model_state_dict.pt', map_location = device)
model.load_state_dict(model_state_dict)

<All keys matched successfully>

In [116]:
import torch
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [117]:
class CustomDataset(Dataset):
    def __init__(self, file_path):
        df = pd.read_csv(file_path)
        self.x1 = df.iloc[:, 0].values
        self.x2 = df.iloc[:, 1].values
        self.x3 = df.iloc[:, 2].values
        self.y = df.iloc[:, 3].values
        self.length = len(df)

    def __getitem__(self, index):
        x = torch.FloatTensor([self.x1[index], self.x2[index], self.x3[index]])
        y = torch.FloatTensor([int(self.y[index])])
        return x, y

    def __len__(self):
        return self.length

In [118]:
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(3, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.layer(x)
        return x

In [119]:
dataset = CustomDataset('./dataset/binary.csv')
dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
val_size = int(dataset_size * 0.1)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], torch.manual_seed(42))
train_dataloader = DataLoader(train_dataset, batch_size = 16, shuffle = True, drop_last = True)
val_dataloader = DataLoader(val_dataset, batch_size = 4, shuffle = True, drop_last = True)
test_dataloader = DataLoader(test_dataset, batch_size = 4, shuffle = False, drop_last = True)

In [120]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CustomModel().to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.SGD(model.parameters(), lr = 0.0001)

In [121]:
for epoch in range(1000):
    cost = 0.0
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        cost += loss
    cost = cost / len(train_dataloader)
    if (epoch + 1) % 100 == 0:
        print(epoch + 1, cost)
        

100 tensor(0.6316, device='cuda:0', grad_fn=<DivBackward0>)
200 tensor(0.6289, device='cuda:0', grad_fn=<DivBackward0>)
300 tensor(0.6225, device='cuda:0', grad_fn=<DivBackward0>)
400 tensor(0.6150, device='cuda:0', grad_fn=<DivBackward0>)
500 tensor(0.6121, device='cuda:0', grad_fn=<DivBackward0>)
600 tensor(0.6033, device='cuda:0', grad_fn=<DivBackward0>)
700 tensor(0.6013, device='cuda:0', grad_fn=<DivBackward0>)
800 tensor(0.5980, device='cuda:0', grad_fn=<DivBackward0>)
900 tensor(0.5941, device='cuda:0', grad_fn=<DivBackward0>)
1000 tensor(0.5868, device='cuda:0', grad_fn=<DivBackward0>)


In [122]:
with torch.no_grad():
    model.eval()
    for x, y in val_dataloader:
        x = x.to(device)
        y = y.to(device)
        outputs = model(x)
        print(outputs)
        print(outputs >= torch.FloatTensor([0.5]).to(device))

tensor([[0.5279],
        [0.6185],
        [0.5403],
        [0.5925]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.4804],
        [0.5484],
        [0.4598],
        [0.4383]], device='cuda:0')
tensor([[False],
        [ True],
        [False],
        [False]], device='cuda:0')
tensor([[0.6504],
        [0.5548],
        [0.6172],
        [0.5434]], device='cuda:0')
tensor([[True],
        [True],
        [True],
        [True]], device='cuda:0')
tensor([[0.6012],
        [0.6783],
        [0.6093],
        [0.4088]], device='cuda:0')
tensor([[ True],
        [ True],
        [ True],
        [False]], device='cuda:0')
tensor([[0.5682],
        [0.5656],
        [0.4469],
        [0.5333]], device='cuda:0')
tensor([[ True],
        [ True],
        [False],
        [ True]], device='cuda:0')
tensor([[0.6006],
        [0.5605],
        [0.5082],
        [0.6639]], device='cuda:0')
tensor([[True],
        [True],
      

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
import torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(1, 2),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(2, 1)
        self._init_weights()
    def _init_weights(self):
        nn.init.xavier_uniform_(self.layer[0].weight)
        self.layer[0].bias.data.fill_(0.01)

        nn.init.xavier_uniform_(self.fc.weight)
        self.fc.bias.data.fill_(0.01)
model = Net().to(device)


In [9]:
for name, param in model.named_parameters():
    print(name, param.data)

layer.0.weight tensor([[-1.3558],
        [ 1.0407]], device='cuda:0')
layer.0.bias tensor([0.0100, 0.0100], device='cuda:0')
fc.weight tensor([[-1.2137, -0.9228]], device='cuda:0')
fc.bias tensor([0.0100], device='cuda:0')


In [11]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(1, 2),
            nn.Sigmoid()
        )
        self.fc = nn.Linear(2, 1)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.constant_(module.bias, 0.01)
        print(module)
model = Net().to(device)


Linear(in_features=1, out_features=2, bias=True)
Sigmoid()
Sequential(
  (0): Linear(in_features=1, out_features=2, bias=True)
  (1): Sigmoid()
)
Linear(in_features=2, out_features=1, bias=True)
Net(
  (layer): Sequential(
    (0): Linear(in_features=1, out_features=2, bias=True)
    (1): Sigmoid()
  )
  (fc): Linear(in_features=2, out_features=1, bias=True)
)


In [12]:
for name, param in model.named_parameters():
    print(name, param.data)

layer.0.weight tensor([[-0.1636],
        [ 0.8895]], device='cuda:0')
layer.0.bias tensor([0.0100, 0.0100], device='cuda:0')
fc.weight tensor([[0.1856, 1.1540]], device='cuda:0')
fc.bias tensor([0.0100], device='cuda:0')


In [13]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
x_data = torch.rand(256, 1) * 10
y_data = 2 * x_data + 1 + torch.randn(256, 1) * 0.5

train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True, drop_last = True)
model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)

In [16]:
for epoch in range(10):
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        _lambda = 0.5
        #l1_loss = sum(p.abs().sum() for p in model.parameters())
        l2_loss = sum(p.pow(2.0).sum() for p in model.parameters())

        #loss = criterion(output, y) + _lambda * l1_loss
        loss = criterion(output, y) + _lambda * l2_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    #print(epoch + 1, loss.item(), l1_loss.item())
    print(epoch + 1, loss.item(), l2_loss.item())

1 2.9241127967834473 4.895062446594238
2 2.9767820835113525 4.866850852966309
3 3.2405450344085693 4.837184429168701
4 3.278742790222168 4.819674491882324
5 3.0985846519470215 4.802468776702881
6 3.4669711589813232 4.784677982330322
7 3.2158541679382324 4.768472194671631
8 2.802398443222046 4.758174419403076
9 3.5105762481689453 4.7433762550354
10 2.9833321571350098 4.735403060913086


In [17]:
print(model.weight.item(), model.bias.item())

2.1590747833251953 -0.25794732570648193


In [ ]:
#model = nn.Linear(1, 1).to(device)
#optimizer = torch.optim.SGD(model.parameters(), lr = 0.01, weight_decay = 0.01)

In [18]:

train_dataset = TensorDataset(x_data, y_data)
train_dataloader = DataLoader(train_dataset, batch_size = 32, shuffle = True, drop_last = True)
model = nn.Linear(1, 1).to(device)
criterion = nn.MSELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)

In [22]:
for epoch in range(20):
    for x, y in train_dataloader:
        x = x.to(device)
        y = y.to(device)

        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
        optimizer.step()

In [23]:
import nltk

In [26]:
for resource in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet = True)


In [27]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, nut we can see plenty ther that needs to be done',
    'It a machine is ecpected  to be infalliable, it cannot also be interlligent'
]

aug = naw.ContextualWordEmbsAug(model_path = 'bert-base-uncased', action = 'insert', device = 'cpu')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(texts)
    print(augmented)

c:\Users\use\Documents\KDT17\컴퓨터비전 심층학습\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


['Those who can imagine anything, can create the impossible.', 'We can only see a short distance ahead, nut we can see plenty ther that needs to be done', 'It a machine is ecpected  to be infalliable, it cannot also be interlligent']
those gentlemen who love can imagine anything, can create completely the perfect impossible.
['Those who can imagine anything, can create the impossible.', 'We can only see a short distance ahead, nut we can see plenty ther that needs to be done', 'It a machine is ecpected  to be infalliable, it cannot also be interlligent']
sorry we can only see some a the short distance of ahead, nut like we can see plenty ther that here needs to be be done
['Those who can imagine anything, can create the impossible.', 'We can only see a short distance ahead, nut we can see plenty ther that needs to be done', 'It a machine is ecpected  to be infalliable, it cannot also be interlligent']
it is a machine is generally ecpected to be called infalliable, for it but cannot als

In [ ]:
import nlpaug.augmenter.char as nac


texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, nut we can see plenty ther that needs to be done',
    'It a machine is ecpected  to be infalliable, it cannot also be interlligent'
]

aug = nac.RandomCharAug(action = 'delete')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)


Those who can imagine anything, can create the impossible.
Tos who can iane anything, can crae the iposile.
We can only see a short distance ahead, nut we can see plenty ther that needs to be done
We can oy see a short istne ahead, nut we can see plny ther ta nds to be on
It a machine is ecpected  to be infalliable, it cannot also be interlligent
It a achn is epece to be nfllble, it cano also be nerigent


In [40]:
import nlpaug.augmenter.char as nac


texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, nut we can see plenty ther that needs to be done',
    'It a machine is ecpected  to be infalliable, it cannot also be interlligent'
]

aug = nac.RandomCharAug(action = 'delete')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)


Those who can imagine anything, can create the impossible.
hos who can imge anyin, can create the mposibl.
We can only see a short distance ahead, nut we can see plenty ther that needs to be done
We can oy see a short dianc ahead, nut we can see lety ther th ned to be de
It a machine is ecpected  to be infalliable, it cannot also be interlligent
It a mcie is ceced to be ifallie, it cnno also be terligen


In [33]:
import nlpaug.augmenter.word as naw


texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, nut we can see plenty ther that needs to be done',
    'It a machine is ecpected  to be infalliable, it cannot also be interlligent'
]

aug = naw.SynonymAug(aug_src = 'wordnet')
augmented_texts = aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)


Those who can imagine anything, can create the impossible.
Those who canful reckon anything, can create the inconceivable.
We can only see a short distance ahead, nut we can see plenty ther that needs to be done
We can solely witness a short aloofness forward, nut we arse see plenty ther that call for to be done
It a machine is ecpected  to be infalliable, it cannot also be interlligent
It a simple machine is ecpected to be infalliable, information technology cannot also be interlligent


In [39]:
import nlpaug.augmenter.word as naw


texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, nut we can see plenty ther that needs to be done',
    'It a machine is ecpected  to be infalliable, it cannot also be interlligent'
]

reserved_tokens = [
    ["can", "can t", "cannot", "could"]
]

reserved_aug = naw.ReservedAug(reserved_tokens = reserved_tokens)
#aug = naw.SynonymAug(aug_src = 'wordnet')
augmented_texts = reserved_aug.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)

Those who can imagine anything, can create the impossible.
Those who could imagine anything, can t create the impossible.
We can only see a short distance ahead, nut we can see plenty ther that needs to be done
We could only see a short distance ahead, nut we cannot see plenty ther that needs to be done
It a machine is ecpected  to be infalliable, it cannot also be interlligent
It a machine is ecpected to be infalliable, it can also be interlligent


In [42]:
import nlpaug.augmenter.word as naw

texts = [
    'Those who can imagine anything, can create the impossible.',
    'We can only see a short distance ahead, nut we can see plenty ther that needs to be done',
    'It a machine is ecpected  to be infalliable, it cannot also be interlligent'
]

back_translation = naw.BackTranslationAug(
    from_model_name = 'facebook/wmt19-en-de',
    to_model_name = 'facebook/wmt19-de-en',
    device = 'cpu'
)

augmented_texts = back_translation.augment(texts)

for text, augmented in zip(texts, augmented_texts):
    print(text)
    print(augmented)


model.safetensors:   1%|          | 10.5M/1.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/825 [00:00<?, ?B/s]

c:\Users\use\Documents\KDT17\컴퓨터비전 심층학습\.venv\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\use\.cache\huggingface\hub\models--facebook--wmt19-de-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/67.0 [00:00<?, ?B/s]

vocab-src.json: 0.00B [00:00, ?B/s]

vocab-tgt.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Those who can imagine anything, can create the impossible.
Anyone who can imagine anything can achieve the impossible.
We can only see a short distance ahead, nut we can see plenty ther that needs to be done
We can only see a short distance ahead, but we can see much more that needs to be done
It a machine is ecpected  to be infalliable, it cannot also be interlligent
It is expected that a machine is infallible, it cannot also be interlligent
